In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "HeartDiseaseTrain-Test.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ketangangal/heart-disease-dataset-uci",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_5396/340940109.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 111k/111k [00:00<00:00, 817kB/s]

First 5 records:    age     sex chest_pain_type  resting_blood_pressure  cholestoral  \
0   52    Male  Typical angina                     125          212   
1   53    Male  Typical angina                     140          203   
2   70    Male  Typical angina                     145          174   
3   61    Male  Typical angina                     148          203   
4   62  Female  Typical angina                     138          294   

      fasting_blood_sugar               rest_ecg  Max_heart_rate  \
0    Lower than 120 mg/ml  ST-T wave abnormality             168   
1  Greater than 120 mg/ml                 Normal             155   
2    Lower than 120 mg/ml  ST-T wave abnormality             125   
3    Lower than 120 mg/ml  ST-T wave abnormality             161   
4  Greater than 120 mg/ml  ST-T wave abnormality             106   

  exercise_induced_angina  oldpeak        slope vessels_colored_by_flourosopy  \
0                      No      1.0  Downsloping                   

In [3]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
def make_factor(variables, cardinalities, values=None):
    size = int(np.prod(cardinalities))
    val = np.zeros(size) if values is None else np.array(values, dtype=float)

    return {
        "var": list(variables),
        "card": list(cardinalities),
        "val": val
    }

In [5]:
def idx_to_assign(idx, cards):
    assign = []

    for c in reversed(cards):
        assign.append(idx % c)
        idx //= c

    return list(reversed(assign))

In [6]:
def assign_to_idx(assign, cards):
    idx = 0
    stride = 1

    for i in range(len(assign)-1, -1, -1):
        idx += assign[i] * stride
        stride *= cards[i]

    return idx

In [7]:
def factor_product(A, B):
    if not A["var"]:
        return B
    if not B["var"]:
        return A

    all_vars = list(dict.fromkeys(A["var"] + B["var"]))

    card_map = {v:c for v,c in zip(A["var"],A["card"])}
    card_map.update({v:c for v,c in zip(B["var"],B["card"])})

    all_cards = [card_map[v] for v in all_vars]

    C = make_factor(all_vars, all_cards)

    for i in range(len(C["val"])):
        assignment = idx_to_assign(i, all_cards)

        amap = {v:assignment[j] for j,v in enumerate(all_vars)}

        ia = assign_to_idx([amap[v] for v in A["var"]], A["card"])
        ib = assign_to_idx([amap[v] for v in B["var"]], B["card"])

        C["val"][i] = A["val"][ia] * B["val"][ib]

    return C

In [8]:
def factor_sum(A, B):
    C = copy.deepcopy(A)
    C["val"] = A["val"] + B["val"]
    return C

In [9]:
def factor_marginalize(F, var):
    if var not in F["var"]:
        return F

    idx = F["var"].index(var)

    new_vars = [v for v in F["var"] if v != var]
    new_cards = [c for i,c in enumerate(F["card"]) if i != idx]

    G = make_factor(new_vars, new_cards if new_cards else [1])

    for i in range(len(F["val"])):
        assignment = idx_to_assign(i, F["card"])

        new_assignment = [assignment[j] for j in range(len(assignment)) if j != idx]

        gi = assign_to_idx(new_assignment, new_cards) if new_cards else 0

        G["val"][gi] += F["val"][i]

    return G

In [10]:
def observe(F, var, val):
    if var not in F["var"]:
        return F

    G = copy.deepcopy(F)
    idx = G["var"].index(var)

    for i in range(len(G["val"])):
        assignment = idx_to_assign(i, G["card"])
        if assignment[idx] != val:
            G["val"][i] = 0

    return G

In [11]:
def calculate_euf(I):
    joint = None

    for rf in I["RandomFactors"]:
        joint = rf if joint is None else factor_product(joint, rf)

    euf = factor_product(joint, I["UtilityFactors"][0])

    decision_vars = set()
    for d in I["DecisionFactors"]:
        decision_vars.update(d["var"])

    for v in list(euf["var"]):
        if v not in decision_vars:
            euf = factor_marginalize(euf, v)

    return euf

In [12]:
def optimize_meu(I):
    D = I["DecisionFactors"][0]
    euf = calculate_euf(I)

    opt = copy.deepcopy(D)
    opt["val"] = np.zeros(len(D["val"]))

    best = np.argmax(euf["val"])
    opt["val"][best] = 1

    return euf["val"][best], opt

In [13]:
def optimize_meu_evidence(I, evidence):
    I2 = copy.deepcopy(I)

    for v,val in evidence.items():
        I2["RandomFactors"] = [
            observe(f, v, val) for f in I2["RandomFactors"]
        ]

    return optimize_meu(I2)

- 0 = D (Disease)
- 1 = C (Chol)
- 2 = E (Exang)
- 3 = A (Age)
- 4 = H (Heart rate)
- 5 = S (Decision)

In [14]:
def discretize(df):

    df = df.copy()

    df["age_bin"] = pd.cut(df["age"], [0,40,55,100], labels=[0,1,2]).astype(int)

    df["chol_bin"] = pd.cut(df["cholestoral"], [0,200,240,1000],
                            labels=[0,1,2]).astype(int)

    df["hr_bin"] = pd.cut(df["Max_heart_rate"], [0,120,160,250],
                         labels=[0,1,2]).astype(int)

    df["exang_bin"] = (df["exercise_induced_angina"] == "Yes").astype(int)

    df["target"] = df["target"].astype(int)

    return df

In [15]:
df = discretize(df)
df.head()

,age,sex,chest_pain_type,resting_blood_pressure,cholestoral,fasting_blood_sugar,rest_ecg,Max_heart_rate,exercise_induced_angina,oldpeak,slope,vessels_colored_by_flourosopy,thalassemia,target,age_bin,chol_bin,hr_bin,exang_bin
0,52,Male,Typical angina,125,212,Lower than 120 mg/ml,ST-T wave abnormality,168,No,1.0,Downsloping,Two,Reversable Defect,0,1,1,2,0
1,53,Male,Typical angina,140,203,Greater than 120 mg/ml,Normal,155,Yes,3.1,Upsloping,Zero,Reversable Defect,0,1,1,1,1
2,70,Male,Typical angina,145,174,Lower than 120 mg/ml,ST-T wave abnormality,125,Yes,2.6,Upsloping,Zero,Reversable Defect,0,2,0,1,1
3,61,Male,Typical angina,148,203,Lower than 120 mg/ml,ST-T wave abnormality,161,No,0.0,Downsloping,One,Reversable Defect,0,2,1,2,0
4,62,Female,Typical angina,138,294,Greater than 120 mg/ml,ST-T wave abnormality,106,No,1.9,Flat,Three,Fixed Defect,0,2,2,0,0


In [16]:
A = make_factor([3], [3])

counts = df["age_bin"].value_counts().sort_index()
A["val"] = (counts.values + 1e-3) / (counts.values + 1e-3).sum()

In [17]:
C = make_factor([1,3], [3,3])  # C | A

for a in range(3):
    sub = df[df["age_bin"] == a]
    total = len(sub) + 3*1e-3

    for c in range(3):
        count = (sub["chol_bin"] == c).sum() + 1e-3
        idx = assign_to_idx([c,a],[3,3])
        C["val"][idx] = count / total

In [18]:
H = make_factor([4,1], [3,3])  # H | C

for c in range(3):
    sub = df[df["chol_bin"] == c]
    total = len(sub) + 3*1e-3

    for h in range(3):
        count = (sub["hr_bin"] == h).sum() + 1e-3
        idx = assign_to_idx([h,c],[3,3])
        H["val"][idx] = count / total

In [19]:
E = make_factor([2,3], [2,3])  # E | A

for a in range(3):
    sub = df[df["age_bin"] == a]
    total = len(sub) + 2*1e-3

    for e in range(2):
        count = (sub["exang_bin"] == e).sum() + 1e-3
        idx = assign_to_idx([e,a],[2,3])
        E["val"][idx] = count / total

In [20]:
D = make_factor([0,1,4,2], [2,3,3,2])  # D | C,H,E

eps = 1e-3

for c in range(3):
    for h in range(3):
        for e in range(2):

            sub = df[
                (df["chol_bin"]==c) &
                (df["hr_bin"]==h) &
                (df["exang_bin"]==e)
            ]

            total = len(sub) + 2*eps
            p1 = (sub["target"].sum() + eps) / total
            p0 = 1 - p1

            i0 = assign_to_idx([0,c,h,e],[2,3,3,2])
            i1 = assign_to_idx([1,c,h,e],[2,3,3,2])

            D["val"][i0] = p0
            D["val"][i1] = p1

In [21]:
S = make_factor([5], [2], [0.5, 0.5])

U = make_factor([0,5], [2,2])
U["val"][assign_to_idx([0,0],[2,2])] = 10
U["val"][assign_to_idx([0,1],[2,2])] = -20
U["val"][assign_to_idx([1,0],[2,2])] = -100
U["val"][assign_to_idx([1,1],[2,2])] = 80

In [22]:
I = {
    "RandomFactors": [A, C, H, E, D],
    "DecisionFactors": [S],
    "UtilityFactors": [U]
}

In [23]:
euf = calculate_euf(I)

best = np.argmax(euf["val"])
print("EU No Test:", euf["val"][0])
print("EU Test   :", euf["val"][1])
print("Optimal   :", "Test" if euf["val"][1] > euf["val"][0] else "No Test")

EU No Test: -45.38081182253087
EU Test   : 30.346192565937166
Optimal   : Test


In [24]:
meu, policy = optimize_meu(I)

print("MEU =", meu)
print("Decision:", np.argmax(policy["val"]))

MEU = 30.346192565937166
Decision: 1


In [26]:
evidence = {
    1: 2,  # chol
    2: 1,  # exang
    3: 1,  # age
    4: 0   # hr
}

meu, policy = optimize_meu_evidence(I, evidence)
print("MEU =", meu)
print("Decision:", np.argmax(policy["val"]))

MEU = 0.038949372216208325
Decision: 0
